<a href="https://colab.research.google.com/github/prateekdhawan/FDE/blob/main/Assignment_3B_Moment_RAG/Moment_RAG_Semantic_vs_Moment_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3B — Semantic Search -> Moment RAG

Baseline semantic-search RAG vs. a **Moment RAG** pipeline over a YouTube transcript, then a
side-by-side comparison.

**Video:** Andrej Karpathy x Stephanie Zhan (Sequoia) — *"From Vibe Coding to Agentic Engineering"*
(`https://www.youtube.com/watch?v=96jN2OCOfLs`, ~30 min).

**Approach (clean A/B):** hold the retriever constant and change only the segmentation
(fixed chunks vs. semantic timestamped *moments*), so any lift is attributable to moments.
Then a **full Moment RAG** arm adds the reference's query-side stack (decompose + hybrid + re-rank).

**Runtime notes:** answer LLM is **Gemini free tier** via its OpenAI-compatible endpoint (no OpenAI key).
Embeddings are **local** `sentence-transformers` (free, no quota). On Colab, set a secret
`GOOGLE_API_KEY`. Free tier is 15 req/min — a retry wrapper handles 429s automatically.

## Setup

In [4]:
!pip install -q youtube-transcript-api sentence-transformers faiss-cpu rank-bm25 fastembed openai

In [5]:
# Config + Gemini client (free tier) + robust retry wrapper
import os, re, json, time, random
import numpy as np
from openai import OpenAI, RateLimitError, InternalServerError, APIConnectionError, APITimeoutError

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
LLM_MODEL = os.getenv("LLM_MODEL", "gemini-3.5-flash-lite")

def _load_key():
    try:
        from google.colab import userdata
        k = userdata.get("GOOGLE_API_KEY")
        if k:
            return k
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")

google_api_key = _load_key()
assert google_api_key, "Set GOOGLE_API_KEY (Colab secret or env var) before running."
client = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

# Free tier throws 429 (rate limit) AND transient 500/503 (model overloaded) — retry BOTH so an
# unattended Run-All survives. Keep the TRUE original on the client so re-running this cell rebinds
# cleanly (no double-wrapping) even after a previous run already patched it.
RETRIABLE = (RateLimitError, InternalServerError, APIConnectionError, APITimeoutError)
if not hasattr(client.chat.completions, "_orig_create"):
    client.chat.completions._orig_create = client.chat.completions.create
_orig_create = client.chat.completions._orig_create

def _create_with_retry(*args, **kwargs):
    last = None
    for attempt in range(6):
        try:
            return _orig_create(*args, **kwargs)
        except RETRIABLE as e:
            last = e
            delay = min(2 ** attempt, 60)            # exponential backoff for 5xx / overload
            m = re.search(r"retry in ([\d.]+)s", str(e))
            if m:                                    # 429 often reports an exact delay — honor it
                delay = float(m.group(1)) + 1
            delay += random.uniform(0, 1.0)          # jitter
            print(f"[{type(e).__name__}] retrying in {delay:.1f}s (attempt {attempt+1}/6)...")
            time.sleep(delay)
    raise last

client.chat.completions.create = _create_with_retry

def llm(prompt, model=None, temperature=0.2):
    # Gemini 400s on a lone system message -> always send as a user turn.
    r = client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return (r.choices[0].message.content or "").strip()

print("Gemini client ready:", LLM_MODEL)

Gemini client ready: gemini-3.5-flash-lite


## Part 0 — Transcript

In [6]:
# Fetch the YouTube transcript (timed caption cues)
from youtube_transcript_api import YouTubeTranscriptApi

VIDEO_ID = "96jN2OCOfLs"

def fetch_transcript(video_id):
    # Prefer youtube-transcript-api v1.x (.fetch); fall back to legacy get_transcript only if it
    # exists, else re-raise the REAL error (e.g. IpBlocked) instead of masking it as AttributeError.
    try:
        fetched = YouTubeTranscriptApi().fetch(video_id)
        return [{"text": s.text, "start": float(s.start), "duration": float(s.duration)} for s in fetched]
    except Exception:
        legacy = getattr(YouTubeTranscriptApi, "get_transcript", None)
        if legacy is None:
            raise
        raw = legacy(video_id)
        return [{"text": s["text"], "start": float(s["start"]), "duration": float(s.get("duration", 0.0))} for s in raw]

cues = fetch_transcript(VIDEO_ID)
_end = cues[-1]["start"] + cues[-1]["duration"]
print(f"{len(cues)} cues, ~{_end/60:.1f} min, ~{sum(len(c['text'].split()) for c in cues)} words")
print("sample:", cues[0])

def fmt_ts(seconds):
    seconds = int(seconds)
    return f"{seconds//60:02d}:{seconds%60:02d}"

966 cues, ~29.7 min, ~6262 words
sample: {'text': "We're so excited for our very first", 'start': 2.08, 'duration': 4.16}


In [7]:
# Local embeddings (free, no API quota) — used by BOTH systems
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

def embed(texts):
    # L2-normalized float32 -> cosine == inner product (FAISS IndexFlatIP).
    if isinstance(texts, str):
        texts = [texts]
    v = embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return v.astype("float32")

print("embedding dim:", embed("hello").shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dim: 384


## Part 1 — Baseline semantic search RAG (fixed-size chunks)

Deliberately naive: fixed ~1000-char overlapping chunks, dense top-k retrieval, Gemini answer.
No timestamps, no hybrid, no re-rank. This is the control arm.

In [8]:
import faiss

def build_fixed_chunks(cues, size=1000, overlap=150):
    chunks, buf, buf_start = [], "", None
    for c in cues:
        if buf_start is None:
            buf_start = c["start"]
        buf += (" " if buf else "") + c["text"].strip()
        if len(buf) >= size:
            chunks.append({"text": buf, "start": buf_start})
            buf, buf_start = buf[-overlap:], c["start"]
    if buf.strip():
        chunks.append({"text": buf, "start": buf_start})
    return chunks

base_chunks = build_fixed_chunks(cues)
print(f"{len(base_chunks)} baseline chunks (~1000 chars each)")

39 baseline chunks (~1000 chars each)


In [9]:
# Dense index over baseline chunks
base_emb = embed([c["text"] for c in base_chunks])
base_index = faiss.IndexFlatIP(base_emb.shape[1])
base_index.add(base_emb)

def dense_search(index, query, k):
    D, I = index.search(embed(query), k)
    return [int(i) for i in I[0]], [float(d) for d in D[0]]

def baseline_retrieve(query, k=4):
    ids, _ = dense_search(base_index, query, k)
    return [base_chunks[i] for i in ids if i != -1]

In [10]:
# Answer synthesis (Gemini) — grounded, cited. Moment answers cite [mm:ss]; baseline cites [n].
def _context_block(items, with_ts):
    lines = []
    for n, it in enumerate(items, 1):
        tag = f"[{fmt_ts(it['start'])}]" if with_ts else f"[{n}]"   # timestamp IS the citation for moments
        head = f"{tag} {it['title']}" if it.get("title") else tag
        lines.append(f"{head}\n{it['text']}")
    return "\n\n".join(lines)

SYNTH_PROMPT = """You are answering a question using ONLY the transcript excerpts below.
Each excerpt starts with a marker in square brackets. Cite the excerpts you use by copying their
exact {marker} marker(s). If the excerpts do not contain the answer, say so plainly. Be concise.

Excerpts:
{context}

Question: {question}

Answer (cite with {marker} markers):"""

def synthesize(question, items, with_ts=False):
    if not items:
        return "No relevant context retrieved."
    marker = "[mm:ss]" if with_ts else "[n]"
    prompt = SYNTH_PROMPT.format(marker=marker, context=_context_block(items, with_ts), question=question)
    return llm(prompt)

def baseline_answer(query, k=4):
    items = baseline_retrieve(query, k)
    return synthesize(query, items, with_ts=False), items

## Part 2 — Moment RAG

A "moment" = a semantically coherent, timestamped span of transcript. We (1) merge tiny caption
cues into sentence-sized units, (2) detect topic-shift boundaries where consecutive-unit
similarity drops, (3) group units into moments, (4) index and retrieve at moment granularity.

In [11]:
# Step 1: merge tiny caption cues into ~sentence-sized units
def merge_cues_to_units(cues, max_words=45, max_secs=20):
    units, buf, start, last_end = [], "", None, None
    for c in cues:
        if start is None:
            start = c["start"]
        buf += (" " if buf else "") + c["text"].strip()
        last_end = c["start"] + c["duration"]
        if len(buf.split()) >= max_words or (last_end - start) >= max_secs:
            units.append({"text": buf, "start": start, "end": last_end})
            buf, start = "", None
    if buf.strip():
        units.append({"text": buf, "start": start if start is not None else cues[-1]["start"], "end": last_end})
    return units

units = merge_cues_to_units(cues)
print(f"{len(units)} units")

130 units


In [12]:
# Step 2: detect topic boundaries (adaptive threshold) and group into moments
unit_emb = embed([u["text"] for u in units])

def segment_into_moments(units, unit_emb, k_std=0.6, min_units=2):
    sims = [float(np.dot(unit_emb[i], unit_emb[i+1])) for i in range(len(units)-1)]
    thr = float(np.mean(sims) - k_std * np.std(sims))
    boundaries, last = [], -min_units
    for i, s in enumerate(sims):
        if s < thr and (i - last) >= min_units:
            boundaries.append(i + 1)
            last = i
    cuts = [0] + boundaries + [len(units)]
    moments = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        seg = units[a:b]
        moments.append({"text": " ".join(u["text"] for u in seg),
                        "start": seg[0]["start"], "end": seg[-1]["end"]})
    return moments

moments = segment_into_moments(units, unit_emb)
print(f"{len(moments)} moments")
for m in moments[:5]:
    print(f"  [{fmt_ts(m['start'])}-{fmt_ts(m['end'])}] {m['text'][:80]}...")

29 moments
  [00:02-01:08] We're so excited for our very first special guest. He has helped build modern AI...
  [01:06-01:29] using agentic tools like Alpha code adjacent things uh for a while, maybe over t...
  [01:28-02:35] the chunks just came out fine. And then I kept asking for more, and just came ou...
  [02:32-03:48] isn't just better software, it's a whole new computing paradigm. And um software...
  [03:46-04:59] out when you want to install Open Claw, you would expect that normally this is a...


In [13]:
# Step 3 (optional): title each moment in ONE Gemini call (quota-friendly)
ADD_MOMENT_TITLES = True

def title_moments(moments):
    if not ADD_MOMENT_TITLES:
        for m in moments:
            m["title"] = None
        return moments
    listing = "\n".join(f"{i}: {m['text'][:220]}" for i, m in enumerate(moments))
    prompt = ('Give each transcript segment a short 3-6 word title. '
              'Return ONLY JSON: {"titles": {"0": "...", "1": "..."}}.\n\n' + listing)
    try:
        data = json.loads(re.search(r"\{.*\}", llm(prompt), re.DOTALL).group())
        titles = data.get("titles", {})
        for i, m in enumerate(moments):
            m["title"] = titles.get(str(i)) or None
    except Exception as e:
        print("titling failed, continuing without titles:", e)
        for m in moments:
            m["title"] = None
    return moments

moments = title_moments(moments)
print("sample titles:", [m.get("title") for m in moments[:5]])

sample titles: ['Welcoming our special guest AI pioneer', 'Using agentic coding tools effectively', 'Building trust in AI coding', 'Software 3.0 new computing paradigm', 'Installing OpenClaw via shell scripts']


In [14]:
# Step 4: index moments (dense) — the CLEAN A/B arm (same retriever as baseline)
moment_emb = embed([m["text"] for m in moments])
moment_index = faiss.IndexFlatIP(moment_emb.shape[1])
moment_index.add(moment_emb)

def moment_dense_retrieve(query, k=4):
    ids, _ = dense_search(moment_index, query, k)
    return [moments[i] for i in ids if i != -1]

def moment_answer_dense(query, k=4):
    items = moment_dense_retrieve(query, k)
    return synthesize(query, items, with_ts=True), items

In [15]:
# Step 5: FULL Moment RAG — decompose -> hybrid (dense + BM25, RRF) -> cross-encoder re-rank
from rank_bm25 import BM25Okapi

def _tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

bm25 = BM25Okapi([_tok(m["text"]) for m in moments])

def _rank_dense(query, n):
    ids, _ = dense_search(moment_index, query, n)
    return [i for i in ids if i != -1]

def _rank_bm25(query, n):
    scores = bm25.get_scores(_tok(query))
    return [int(i) for i in np.argsort(scores)[::-1][:n]]

def rrf_fuse(rankings, k=60):
    agg = {}
    for ranking in rankings:
        for rank, d in enumerate(ranking):
            agg[d] = agg.get(d, 0.0) + 1.0 / (k + rank + 1)
    return sorted(agg, key=lambda d: agg[d], reverse=True)

try:
    from fastembed.rerank.cross_encoder import TextCrossEncoder
    _reranker = TextCrossEncoder(model_name="Xenova/ms-marco-MiniLM-L-6-v2")
    RERANK_OK = True
except Exception as e:
    print("re-ranker unavailable, using fused order:", e)
    RERANK_OK = False

def decompose(query):
    prompt = ('If the question has multiple parts, split it into sub-questions; else keep one. '
              'Return ONLY JSON: {"subQuestions": ["..."]}.\nQuestion: ' + query)
    try:
        subs = json.loads(re.search(r"\{.*\}", llm(prompt), re.DOTALL).group()).get("subQuestions", [])
        subs = [s.strip() for s in subs if isinstance(s, str) and s.strip()]
        return subs or [query]
    except Exception:
        return [query]

def full_moment_retrieve(query, k=4, pool=10):
    subs = decompose(query)
    rankings = []
    for sq in subs:
        rankings.append(_rank_dense(sq, pool))
        rankings.append(_rank_bm25(sq, pool))
    fused = rrf_fuse(rankings)[: max(k, 8)]
    if RERANK_OK and fused:
        scores = list(_reranker.rerank(query, [moments[i]["text"] for i in fused]))
        fused = [d for _, d in sorted(zip(scores, fused), reverse=True)]
    return [moments[i] for i in fused[:k]], subs

def moment_answer_full(query, k=4):
    items, subs = full_moment_retrieve(query, k)
    return synthesize(query, items, with_ts=True), items, subs

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

## Part 3 — Comparison

Same query set through baseline vs. Moment RAG. First the clean A/B (baseline vs. moment-dense,
isolating the segmentation effect), then the full Moment RAG showcase with an optional LLM judge.

In [16]:
QUERIES = [
    "What does Karpathy say about vibe coding?",
    "How does he describe the role of agents in software engineering?",
    "What did they discuss about reinforcement learning environments?",
    "What is Karpathy's view on the near-term future of AI?",
    "What advice is given to startups or builders?",
    "Where do Karpathy and the host push back on each other?",
]

def show_ab(query):
    print("=" * 90)
    print("Q:", query)
    b_ans, b_items = baseline_answer(query)
    m_ans, m_items = moment_answer_dense(query)
    print("\n-- BASELINE retrieved (fixed chunks) --")
    for n, it in enumerate(b_items, 1):
        print(f"  [{n}] {it['text'][:90]}...")
    print("\n-- MOMENT retrieved (semantic moments + timestamps) --")
    for n, it in enumerate(m_items, 1):
        t = f" {it.get('title')}" if it.get("title") else ""
        print(f"  [{n}] ({fmt_ts(it['start'])}-{fmt_ts(it['end'])}){t}: {it['text'][:70]}...")
    print("\n-- BASELINE answer --\n", b_ans)
    print("\n-- MOMENT (clean A/B) answer --\n", m_ans)

# Runs all 6 queries; the retry wrapper absorbs free-tier 429s (just slower).
for q in QUERIES:
    show_ab(q)

Q: What does Karpathy say about vibe coding?

-- BASELINE retrieved (fixed chunks) --
  [1] nt, some things easier than others. Um because even for like things that are like writing ...
  [2]  that? Was that feeling exhilarating or unsettling? Uh yeah, mixture of both for sure. Uh ...
  [3] credible. But then I would say agentic engineering is about preserving the quality bar of ...
  [4] We're so excited for our very first special guest. He has helped build modern AI, then exp...

-- MOMENT retrieved (semantic moments + timestamps) --
  [1] (15:35-17:28) Everything is automatable and vibe coding: Um So, I I do think that ultimately um Uh yeah, I think uh Everything....
  [2] (00:02-01:08) Welcoming our special guest AI pioneer: We're so excited for our very first special guest. He has helped build...
  [3] (01:28-02:35) Building trust in AI coding: the chunks just came out fine. And then I kept asking for more, and ju...
  [4] (24:04-24:49) Building and evaluating AI models: you build

In [17]:
# Full Moment RAG showcase (decompose + hybrid + re-rank) + optional LLM judge
RUN_LLM_JUDGE = True

def judge(query, ans_a, ans_b):
    prompt = ("Question: " + query + "\n\nAnswer A:\n" + ans_a + "\n\nAnswer B:\n" + ans_b +
              '\n\nWhich answer is better grounded and more complete? '
              'Return ONLY JSON: {"winner": "A", "reason": "one line"}  (winner in A|B|tie).')
    try:
        return json.loads(re.search(r"\{.*\}", llm(prompt), re.DOTALL).group())
    except Exception:
        return {"winner": "?", "reason": "judge parse failed"}

tally = {"A": 0, "B": 0, "tie": 0, "?": 0}
for q in QUERIES:
    print("=" * 90)
    print("Q:", q)
    b_ans, _ = baseline_answer(q)
    f_ans, f_items, subs = moment_answer_full(q)
    print("sub-questions:", subs)
    print("\n-- FULL Moment RAG retrieved --")
    for n, it in enumerate(f_items, 1):
        t = f" {it.get('title')}" if it.get("title") else ""
        print(f"  [{n}] ({fmt_ts(it['start'])}-{fmt_ts(it['end'])}){t}")
    print("\n-- FULL Moment RAG answer --\n", f_ans)
    if RUN_LLM_JUDGE:
        v = judge(q, b_ans, f_ans)
        tally[v.get("winner", "?")] = tally.get(v.get("winner", "?"), 0) + 1
        print("\nJUDGE (A=baseline, B=full Moment RAG):", v)
print("\nJudge tally (A=baseline, B=full Moment RAG):", tally)

Q: What does Karpathy say about vibe coding?
sub-questions: ['What does Karpathy say about vibe coding?']

-- FULL Moment RAG retrieved --
  [1] (15:35-17:28) Everything is automatable and vibe coding
  [2] (00:02-01:08) Welcoming our special guest AI pioneer
  [3] (01:28-02:35) Building trust in AI coding
  [4] (17:26-19:31) Generational shifts in ChatGPT usage

-- FULL Moment RAG answer --
 Karpathy states that vibe coding raises the floor for everyone in terms of what they can do in software, allowing everyone to vibe code anything [15:35]. He also describes how he personally started vibe coding after trusting the system more and more as code chunks continued to come out fine without needing corrections [01:28].
[RateLimitError] retrying in 39.6s (attempt 1/6)...

JUDGE (A=baseline, B=full Moment RAG): {'winner': 'A', 'reason': 'Answer A provides a more comprehensive and well-structured summary including multiple bullet points and contrasting agentic engineering.'}
Q: How does he de

## Findings, limitations, improvements

*Observations from this run — Karpathy × Zhan, 29.7 min → 39 baseline chunks vs. 29 timestamped moments; all 6 queries; moment answers cite inline `[mm:ss]`.*

**What we saw**
- **Judge tally (full Moment RAG vs. baseline): 3 wins – 2 – 1 tie.** Moment RAG is ahead, but it's not a clean sweep — which is the honest result.
- **Where moments clearly win:**
  - *"Advice to startups/builders"* — the **baseline answered "the transcript does not contain advice"**, while Moment RAG surfaced concrete advice: solve tractable, verifiable problems `[13:21]`; get the most out of your coding tools `[17:26]`. Better segmentation retrieved content the baseline missed entirely.
  - *"RL environments"* — the moment answer was substantially fuller (verifiable settings → RL environments with verification rewards → why models peak in math/code → labs' economic influence, `[14:19]`, `[08:12]`) vs. a thin baseline answer.
- **Where the baseline wins / ties (kept honestly):** on *"vibe coding"* and *"role of agents"* the judge preferred the baseline's tighter, more structured summary; on *"near-term future of AI"* the pure moment-dense arm **missed** the span and answered "the excerpts do not contain the answer," while the baseline found it. On *"where do they push back"* both correctly returned "not in the transcript" (tie).
- **The query-side stack matters:** on *"near-term future"* the full arm (decompose → dense+BM25 RRF → cross-encoder re-rank) retrieved differently from pure moment-dense and the judge preferred it — hybrid + re-rank partly recovered a case where moment-dense alone failed.
- **Timestamps are the real payoff:** every Moment answer cites clickable `[mm:ss]` deep-links; the baseline fundamentally cannot say *where* in the video the content is.

**Limitations**
- Retrieval quality is bounded by **local MiniLM embeddings** (384-dim) — weaker than the reference's OpenAI `text-embedding-3-large@1024`; moment-dense missed one answerable query.
- Semantic-boundary detection is **threshold-sensitive** (`mean − 0.6·std`); adjacent moments overlap ~2s because the caption cues themselves overlap.
- **No labelled ground truth** — this is qualitative evidence plus an LLM-judge that is itself noisy (e.g. it rewarded a cautious "not fully answerable" reply over a confident baseline one).

**Possible improvements**
- HyDE-at-ingestion (pre-tag each moment with the questions it answers) as a 3rd retrieval branch, as in the reference.
- Speaker-aware / overlap-trimmed moment boundaries; a stronger embedder or re-ranker; raise moment-dense recall (larger `k`, merge near-duplicate moments).
- A small labelled query set for quantitative recall/precision.

**Substitutions vs. the reference (traversaal-ai/momentsearch):** OpenAI embeddings → local MiniLM; OpenAI synth → Gemini free tier; Qdrant → FAISS + rank_bm25 + RRF; FastAPI app/UI → this notebook; deployment (Docker/Fly) out of scope (that is the separate ARGUS assignment).